# Task 2 (Easy) - Content Type Prediction Model
**Auspify Technologies - Machine Learning Internship**

Predicts whether a Netflix title is a Movie or a TV Show using its
available metadata (genres, rating, country, release year, duration).

Workflow:

  Step 1: Select relevant dataset features.
  
  Step 2: Encode categorical variables.
  
  Step 3: Train classification models.
  
  Step 4: Evaluate prediction performance.
  
  Step 5: Compare model accuracy.

Note on 'duration': the raw duration string ("90 min" vs "2 Seasons") would
leak the answer directly, since only Movies use "min" and only TV Shows use
"Season(s)". To keep this a genuine modeling exercise, we extract just the
*numeric* duration value (e.g. 90, or 2) and drop the unit text -- the model
has to learn from the distribution of values, not a literal keyword.

In [1]:
import re
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

DATA_PATH = "Dataset.csv"
RANDOM_STATE = 42

def load_and_select_features(path):
    """Step 1: select & engineer features."""
    df = pd.read_csv(path)
    for col in ["director", "country", "listed_in", "rating"]:
        df[col] = df[col].fillna("")

    df["duration_value"] = df["duration"].apply(
        lambda x: int(re.search(r"\d+", x).group()) if re.search(r"\d+", x) else 0
    )
    df["genre_count"] = df["listed_in"].apply(lambda x: len(x.split(",")) if x else 0)
    # FIX: compare against the literal placeholder "Not Given", not ""
    df["has_director"] = (df["director"] != "Not Given").astype(int)
    df["primary_country"] = df["country"].apply(lambda x: x.split(",")[0].strip() if x else "Unknown")
    df["is_international"] = (df["country"].str.contains(",")).astype(int)
    return df, df["type"]

def encode(features, cat_cols):
    """Step 2: encode categorical variables."""
    encoded = features.copy()
    for col in cat_cols:
        encoded[col] = LabelEncoder().fit_transform(encoded[col])
    return encoded

def train_and_eval(X, y, label):
    """Steps 3-4: train Logistic Regression / Decision Tree / Random Forest and evaluate."""
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
    )
    models = {
        "Logistic Regression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
        "Decision Tree": DecisionTreeClassifier(random_state=RANDOM_STATE),
        "Random Forest": RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE),
    }
    print(f"=== {label} ===")
    print(f"Features used: {list(X.columns)}")
    results = {}
    for name, model in models.items():
        model.fit(X_train, y_train)
        acc = accuracy_score(y_test, model.predict(X_test))
        results[name] = acc
        print(f"  {name:22s}: accuracy={acc:.4f}")
    best = max(results.items(), key=lambda x: x[1])
    print(f"  -> Best: {best[0]} ({best[1]:.4f})")
    return results

## Step 1–2: Load data and select/encode features

In [2]:
df, target = load_and_select_features(DATA_PATH)
y = LabelEncoder().fit_transform(target)
print(f"Target classes: {sorted(target.unique().tolist())}")
print(f"Class balance: {target.value_counts(normalize=True).round(3).to_dict()}")

Target classes: ['Movie', 'TV Show']
Class balance: {'Movie': 0.697, 'TV Show': 0.303}


## Step 3–4: Model A - Preventing Data Leakage in the Original Approach

In [3]:
features_leak = df[["rating", "primary_country", "release_year",
                    "duration_value", "genre_count", "has_director"]].copy()
Xa = encode(features_leak, ["rating", "primary_country"])
baseline_acc = target.value_counts(normalize=True).max()
print(f"Majority-class baseline accuracy: {baseline_acc:.4f}\n")
res_a = train_and_eval(Xa, y, "Model A: WITH duration_value (data leakage)")

Majority-class baseline accuracy: 0.6969

=== Model A: WITH duration_value (data leakage) ===
Features used: ['rating', 'primary_country', 'release_year', 'duration_value', 'genre_count', 'has_director']
  Logistic Regression   : accuracy=0.9983
  Decision Tree         : accuracy=0.9989
  Random Forest         : accuracy=0.9977
  -> Best: Decision Tree (0.9989)


## Step 3–4: Model B - genuine features only

In [4]:
features_clean = df[["rating", "primary_country", "release_year",
                     "genre_count", "has_director", "is_international"]].copy()
Xb = encode(features_clean, ["rating", "primary_country"])
res_b = train_and_eval(Xb, y, "Model B: WITHOUT duration_value (genuine features)")

=== Model B: WITHOUT duration_value (genuine features) ===
Features used: ['rating', 'primary_country', 'release_year', 'genre_count', 'has_director', 'is_international']
  Logistic Regression   : accuracy=0.9482
  Decision Tree         : accuracy=0.9369
  Random Forest         : accuracy=0.9397
  -> Best: Logistic Regression (0.9482)


## Step 5: Final comparison

In [5]:
print(f"Majority-class baseline : {baseline_acc:.4f}")
print(f"Model A best (leaky)    : {max(res_a.values()):.4f}  <- inflated by duration leakage")
print(f"Model B best (genuine)  : {max(res_b.values()):.4f}  <- honest measure using metadata only")

Majority-class baseline : 0.6969
Model A best (leaky)    : 0.9989  <- inflated by duration leakage
Model B best (genuine)  : 0.9482  <- honest measure using metadata only


## Conclusion
The honest, leakage-free model reaches **~94.8% accuracy** (vs. a 69.7%
majority-class baseline) using only rating, country, release year, genre
count, director presence and international-coproduction status, a solid,
defensible result. The 99.8%+ figure from the original submission mostly
reflects a units artifact in `duration`, not model skill.